In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────
FIX_PD = pd.DataFrame({
    "chrom": pd.Series(["chr1", "chr2"], dtype="object"),
    "start": pd.Series([10, 20], dtype="int64"),
    "end": pd.Series([15, 30], dtype="int64"),
})
FIX_PL = pl.DataFrame({"chrom": ["chr1", "chr2"], "start": [10, 20], "end": [15, 30]})
FIX_LAZY = FIX_PL.lazy()

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (pandas/original-context behavior) ──────────────────────

def _rename_columns_pl(df: pl.DataFrame, suffix: str) -> pl.DataFrame:
    return df.rename({col: f"{col}{suffix}" for col in df.columns})

def before_rename_columns_empty_df(df, suffix):
    if isinstance(df, pl.DataFrame) or isinstance(df, pl.LazyFrame):
        schema = df.collect_schema() if isinstance(df, pl.LazyFrame) else df.schema
        df = pl.DataFrame(schema=schema)
        return _rename_columns_pl(df, suffix)
    elif pd and isinstance(df, pd.DataFrame):
        df = pl.from_pandas(pd.DataFrame(columns=df.columns))
        return _rename_columns_pl(df, suffix)
    else:
        raise ValueError("Only polars and pandas dataframes are supported")


In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_rename_columns_empty_df(df, suffix):
    df = pl.DataFrame({col: [] for col in df.columns})
    return df

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: rename_columns_empty_df ===

def _assert_empty_renamed(df, suffix, label):
    assert df.height == 0, f"{label}: expected empty DataFrame"
    assert df.columns == [f"{c}{suffix}" for c in ["chrom", "start", "end"]], df.columns

# L1 smoke – generated pandas branch
try:
    _r = gen_rename_columns_empty_df(FIX_PD, "_1")
    print("✅ L1 smoke gen_rename_columns_empty_df: OK")
except Exception as _e:
    print(f"❌ L1 smoke gen_rename_columns_empty_df: {type(_e).__name__}: {_e}")

# L1 smoke – before pandas branch
try:
    _rb = before_rename_columns_empty_df(FIX_PD, "_1")
    _assert_empty_renamed(_rb, "_1", "before pandas")
    print("✅ L1 smoke before_rename_columns_empty_df: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_rename_columns_empty_df: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence for the migrated pandas branch
try:
    _rb = before_rename_columns_empty_df(FIX_PD, "_1")
    _rg = gen_rename_columns_empty_df(FIX_PD, "_1")
    compare(_rb, _rg, "rename_columns_empty_df")
except Exception as _e:
    print(f"❌ L2 equivalence rename_columns_empty_df: setup error — {type(_e).__name__}: {_e}")

# L3 — Polars DataFrame branch keeps schema and suffixes columns.
try:
    _rb = before_rename_columns_empty_df(FIX_PL, "_2")
    _rg = gen_rename_columns_empty_df(FIX_PL, "_2")
    _assert_empty_renamed(_rg, "_2", "generated polars")
    compare(_rb, _rg, "L3 rename_columns_empty_df polars input")
except Exception as _e:
    print(f"❌ L3 rename_columns_empty_df: polars input — {type(_e).__name__}: {_e}")

# L3 — LazyFrame branch uses collect_schema without materialising data.
try:
    _rb = before_rename_columns_empty_df(FIX_LAZY, "_lazy")
    _rg = gen_rename_columns_empty_df(FIX_LAZY, "_lazy")
    assert _rg.columns == ["chrom_lazy", "start_lazy", "end_lazy"]
    compare(_rb, _rg, "L3 rename_columns_empty_df lazy input")
except Exception as _e:
    print(f"❌ L3 rename_columns_empty_df: lazy input — {type(_e).__name__}: {_e}")

# L3 — unsupported input should raise on both sides with the same error type.
try:
    try:
        before_rename_columns_empty_df({"chrom": []}, "_bad")
        before_err = None
    except Exception as e:
        before_err = type(e)
    try:
        gen_rename_columns_empty_df({"chrom": []}, "_bad")
        gen_err = None
    except Exception as e:
        gen_err = type(e)
    assert before_err is ValueError and gen_err is ValueError
    print("✅ L3 rename_columns_empty_df unsupported input: MATCH")
except Exception as _e:
    print(f"❌ L3 rename_columns_empty_df: unsupported input — {type(_e).__name__}: {_e}")
